# Query 6: Window Functions

Run `ddl.ipynb` and `dml.ipynb` first. Window functions calculate across related rows while keeping each result row visible.

In [ ]:
%load_ext sql
%sql sqlite:///food-analysis/food_analysis.db

## Concepts

- `OVER` defines the rows a window function can use.
- `PARTITION BY` starts a separate calculation for each group.
- `ORDER BY` defines the order used by ranking and running calculations.
- Unlike `GROUP BY`, a window function does not collapse the rows in its input.

## Rank customers by delivered-order spend

The inner query calculates one row per customer. `RANK` orders those totals; tied totals receive the same rank.

In [ ]:
%%sql
SELECT customer_id, customer_name, delivered_orders, total_spend,
       RANK() OVER (ORDER BY total_spend DESC) AS spend_rank
FROM (
    SELECT c.customer_id, c.customer_name,
           COUNT(o.order_id) AS delivered_orders,
           ROUND(SUM(o.order_amount), 2) AS total_spend
    FROM customers AS c
    JOIN orders AS o ON o.customer_id = c.customer_id
    WHERE o.status = 'delivered'
    GROUP BY c.customer_id, c.customer_name
) AS customer_totals
ORDER BY spend_rank, customer_id
LIMIT 10;

## Show each customer's two most recent orders

`ROW_NUMBER` starts at 1 for each customer because of `PARTITION BY`. The outer query filters to the first two rows per customer.

In [ ]:
%%sql
SELECT customer_id, order_id, order_placed_at, status, order_number
FROM (
    SELECT customer_id, order_id, order_placed_at, status,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY order_placed_at DESC, order_id DESC
           ) AS order_number
    FROM orders
) AS ranked_orders
WHERE order_number <= 2
ORDER BY customer_id, order_number
LIMIT 20;

## Calculate a running total of delivered order amounts by month

The inner query creates monthly totals; the windowed `SUM` accumulates them in month order.

In [ ]:
%%sql
SELECT order_month, monthly_amount,
       ROUND(SUM(monthly_amount) OVER (
           ORDER BY order_month
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ), 2) AS running_amount
FROM (
    SELECT strftime('%Y-%m', order_placed_at) AS order_month,
           SUM(order_amount) AS monthly_amount
    FROM orders
    WHERE status = 'delivered'
    GROUP BY strftime('%Y-%m', order_placed_at)
) AS monthly_totals
ORDER BY order_month;